In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
sample_submission = pd.read_csv('/kaggle/input/california-house-prices/sample_submission.csv')
train_data = pd.read_csv('/kaggle/input/california-house-prices/train.csv')
test_data = pd.read_csv('/kaggle/input/california-house-prices/test.csv')

In [ ]:
sample_submission.shape, train_data.shape, test_data.shape

In [ ]:
df=pd.read_csv('/kaggle/input/california-house-prices/train.csv')
df.info()

In [ ]:
df_train=df.copy()
df_test=pd.read_csv('/kaggle/input/california-house-prices/test.csv')
for field in ['Listed On', 'Last Sold On']:
    df_train[field]=pd.to_datetime(df[field])
    df_test[field]=pd.to_datetime(df_test[field])

In [ ]:
cate_cols = []
num_cols = []
date_cols = []
dtypes = df_train.dtypes
for col, dtype in dtypes.items():
    if dtype=='object':
        cate_cols.append(col)
    elif dtype.name.startswith('datetime'):
        date_cols.append(col)
    else:
        num_cols.append(col)

In [ ]:
id_col = 'Id'
target_col = 'Sold Price'

for col in [id_col, target_col]:
    num_cols.remove(col)
print(cate_cols)
print(num_cols)
print(date_cols)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from pandas.api.types import is_string_dtype, is_numeric_dtype

class Num_Features(BaseEstimator, TransformerMixin):
    def __init__(self, cols = [], fillna = False, addna = False):
        self.fillna = fillna
        self.cols = cols
        self.addna = addna
        self.na_cols = []
        self.imputers = {}
    def fit(self, X, y=None):
        for col in self.cols:
            if self.fillna:
                self.imputers[col] = X[col].median()
            if self.addna and X[col].isnull().sum():
                self.na_cols.append(col)
        print(self.na_cols, self.imputers)
        return self
    def transform(self, X, y=None):
        df = X.loc[:, self.cols]
        for col in self.imputers:
            df[col].fillna(self.imputers[col], inplace=True)
        for col in self.na_cols:
            df[col+'_na'] = pd.isnull(df[col])
        return df

In [ ]:
class Imputer(BaseEstimator, TransformerMixin):
    def __init__(self, strategy, fill_value):
        self.strategy = strategy
        self.fill_value = fill_value
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        for col, content in X.items():
            X[col].fillna(self.fill_value, inplace=True)
        return X

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, LabelBinarizer, StandardScaler
from sklearn.impute import SimpleImputer
num_pipeline = Pipeline([
    ('select_num', Num_Features(cols=num_cols, fillna='median', addna=True)),
])
X_num = num_pipeline.fit_transform(df_train)

In [ ]:
class CatEncoder(BaseEstimator, TransformerMixin):
    def __init__(self,cols, max_n_cat=7, onehot_cols=[], orders={}):
        self.cols = cols
        self.onehot_cols=onehot_cols
        self.cats = {}
        self.max_n_cat = max_n_cat
        self.orders = orders
    def fit(self, X, y=None):
        df_cat =  X.loc[:, self.cols]
        for n,c in df_cat.items():
            df_cat[n].fillna('NAN', inplace=True)
            df_cat[n] = c.astype('category').cat.as_ordered()
            if n in self.orders:
                df_cat[n].cat.set_categories(self.orders[n], ordered=True, inplace=True)
            cats_count = len(df_cat[n].cat.categories)
            if cats_count<=2 or cats_count>self.max_n_cat:
                self.cats[n] = df_cat[n].cat.categories
                if n in self.onehot_cols:
                    self.onehot_cols.remove(n)
            elif n not in self.onehot_cols:
                self.onehot_cols.append(n)

        print(self.onehot_cols)
        return self
    def transform(self, df, y=None):
        X = df.loc[:, self.cols]
        for col in self.cats:
            X[col].fillna('NAN', inplace=True)
            X.loc[:,col] = pd.Categorical(X[col], categories=self.cats[col], ordered=True)
            X.loc[:,col] = X[col].cat.codes

#         for n,c in X.items():
#             if n in self.cats:
#                 X[n] = pd.Categorical(c, categories=self.cats[n], ordered=True)
#                 X[n] = X[n].cat.codes + 1
#             else:
#                 X[n] = c.astype('category').cat.as_ordered()
        if len(self.onehot_cols):
            df_1h = pd.get_dummies(X[self.onehot_cols], dummy_na=True)
            df_drop=X.drop(self.onehot_cols,axis=1)
            return pd.concat([df_drop, df_1h], axis=1)

        return X

In [ ]:
cat_pipeline = Pipeline([
    ('cat_encoder', CatEncoder(cols=cate_cols))
])
X_cate = cat_pipeline.fit_transform(df_train)

In [ ]:
def add_datepart(df, field_name, prefix=None, drop=True, time=False):
    field = df[field_name]
    if prefix is None:
        prefix = re.sub('[Dd]ate$', '', field_name)
    attr = ['Year', 'Month', 'Week', 'Day', 'Dayofweek', 'Dayofyear', 'Is_month_end', 'Is_month_start', 'Is_quarter_end', 'Is_quarter_start', 'Is_year_end', 'Is_year_start']
    if time: attr = attr + ['Hour', 'Minute', 'Second']
    # Pandas removed `dt.week` in v1.1.10
    week = field.dt.isocalendar().week.astype(field.dt.day.dtype) if hasattr(field.dt, 'isocalendar') else field.dt.week
    for n in attr: df[prefix + n] = getattr(field.dt, n.lower()) if n != 'Week' else week
    mask = ~field.isna()
    df[prefix + 'Elapsed'] = np.where(mask,field.values.astype(np.int64) // 10 ** 9,np.nan)
    if drop: df.drop(field_name, axis=1, inplace=True)
    return df

In [ ]:
import re
class Datepart(BaseEstimator, TransformerMixin):
    def __init__(self, cols, time=False):
        self.cols = cols
        self.time = time
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        df_dates = X.loc[:, self.cols]
        for col in self.cols:
            add_datepart(df_dates, col, time=False)
        return df_dates
    
date_pipeline = Pipeline([
    ('datepart', Datepart(cols=date_cols)),
    ('imputer', Imputer(strategy="constant", fill_value=-1)),
])

In [ ]:
X_date = date_pipeline.fit_transform(df_train)

In [ ]:
y_train = np.log(df_train[target_col])
X_train = pd.concat([X_num, X_cate,X_date], axis=1)
X_train.shape, y_train.shape

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import lightgbm as lgb
lgbmodel = lgb.LGBMRegressor()

param_grid = {
    'boosting_type': ['gbdt'],
    'objective': ['regression'],
    'metric': ['auc'],
    'nthread':[4],
    "n_estimators" : np.arange(50, 300, 50),
    "learning_rate" :[0.01,0.05,0.1,0.3],
    "max_depth": np.arange(2,8,1),
    'colsample_bytree':[0.6,0.8,1],
    'lambda_l1': [1e-5,1e-3,1e-1,0.0,0.1,0.3,0.5,0.7,0.9,1.0],
    'lambda_l2': [1e-5,1e-3,1e-1,0.0,0.1,0.3,0.5,0.7,0.9,1.0],
    'feature_fraction': [0.6,0.7,0.8,0.9,1.0],
    'bagging_fraction': [0.6,0.7,0.8,0.9,1.0],
    'bagging_freq': np.arange(0,81,10),
    'max_bin': np.arange(5,256,10),
    'min_data_in_leaf':np.arange(1,101,10),
    'num_leaves':np.arange(5, 100, 5),
    'min_split_gain':[0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0],
}
lgbmodel = RandomizedSearchCV(estimator = lgbmodel ,
                          param_distributions = param_grid,
                          n_iter = 100,
                          verbose=1,
                          n_jobs = -1,
                          cv = 5)
lgbmodel.fit(X_train,y_train)
lgbmodel.best_score_

In [ ]:
lgbmodel.best_estimator_.get_params()

In [ ]:
modellgb = lgb.LGBMRegressor(boosting_type='gbdt',objective='regression',colsample_bytree=1,metrics='auc',learning_rate=lgbmodel.best_estimator_.get_params()['learning_rate'], n_estimators=lgbmodel.best_estimator_.get_params()['n_estimators'], max_depth=lgbmodel.best_estimator_.get_params()['max_depth'],num_leaves = lgbmodel.best_estimator_.get_params()['num_leaves'],max_bin = lgbmodel.best_estimator_.get_params()['max_bin'],min_child_samples=lgbmodel.best_estimator_.get_params()['min_child_samples'],min_child_weight=0.001,min_split_gain=lgbmodel.best_estimator_.get_params()['min_split_gain'],min_data_in_leaf = lgbmodel.best_estimator_.get_params()['min_data_in_leaf'], bagging_fraction = lgbmodel.best_estimator_.get_params()['bagging_fraction'],feature_fraction = lgbmodel.best_estimator_.get_params()['feature_fraction'],bagging_freq=lgbmodel.best_estimator_.get_params()['bagging_freq'],lambda_l1=lgbmodel.best_estimator_.get_params()['lambda_l1'],lambda_l2=lgbmodel.best_estimator_.get_params()['lambda_l2'])
  
modellgb.fit(X_train,y_train)


# Feature Selection
# # feature importance

In [ ]:
def rf_feat_importance(modellgb, df):
    return pd.DataFrame({'cols':df.columns, 'imp':modellgb.feature_importances_}).sort_values('imp', ascending=False)

fi = rf_feat_importance(modellgb, X_train)
fi[:50]

In [ ]:
import shap  #将博弈论与特征权重增益组合  SHAP值是在这些特征之间的公平的信用分配，并且具有博弈论一致性的理论保证，这使得它们通常比整个数据集中的那些典型特征的重要性更值得信赖。
shap.initjs()   

explainer = shap.TreeExplainer(modellgb)
shap_values = explainer(X_train)
shap.plots.waterfall(shap_values[0])

In [ ]:
shap.plots.beeswarm(shap_values,max_display=18)

In [ ]:
shap.plots.bar(shap_values,max_display=19)

In [ ]:
# 最好提交的特征，18个
to_keep_final=['Listed Price', 
               'Tax assessed value',
               'Annual tax amount',
               'Listed OnElapsed',
               'Last Sold Price', 
               'Zip', 
               'Parking', 
               'Year built',
               'Total interior livable area', 
               'Type',
               'Elementary School Score',
               'Listed OnYear',
               'Elementary School Distance',
               'Last Sold OnElapsed',
               'Middle School Score',
               'Lot',
               'Appliances included',
               'Listed OnDayofyear']
# to_keep_final=['Listed Price', 'Tax assessed value', 'Last Sold Price', 'Zip', 'Total interior livable area', 'Listed OnElapsed', 'Elementary School Score', 'Last Sold OnElapsed', 'Year built', 'Listed OnYear', 'High School Distance', 'Lot', 'Parking', 'Middle School Score', 'Elementary School Distance', 'Region', 'Bedrooms', 'High School Score', 'Heating', 'Appliances included', 'Flooring', 'Middle School Distance']
X_train_final = X_train[to_keep_final].copy()

In [ ]:
#2nd  model train
from sklearn.model_selection import RandomizedSearchCV
import lightgbm as lgb
lgbmodel = lgb.LGBMRegressor()

param_grid = {
    'boosting_type': ['gbdt'],
    'objective': ['regression'],
    'metric': ['auc'],
    'nthread':[4],
    "n_estimators" : np.arange(50, 300, 50),
    "learning_rate" :[0.01,0.05,0.1,0.3],
    "max_depth": np.arange(2,8,1),
    'colsample_bytree':[0.6,0.8,1],
    'lambda_l1': [1e-5,1e-3,1e-1,0.0,0.1,0.3,0.5,0.7,0.9,1.0],
    'lambda_l2': [1e-5,1e-3,1e-1,0.0,0.1,0.3,0.5,0.7,0.9,1.0],
    'feature_fraction': [0.6,0.7,0.8,0.9,1.0],
    'bagging_fraction': [0.6,0.7,0.8,0.9,1.0],
    'bagging_freq': np.arange(0,81,10),
    'max_bin': np.arange(5,256,10),
    'min_data_in_leaf':np.arange(1,101,10),
    'num_leaves':np.arange(5, 100, 5),
    'min_split_gain':[0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
}
lgbmodel = RandomizedSearchCV(estimator = lgbmodel ,
                          param_distributions = param_grid,
                          n_iter = 100,
                          verbose=1,
                          n_jobs = -1,
                          cv = 5)
lgbmodel.fit(X_train_final,y_train)
lgbmodel.best_score_

In [ ]:
lgbmodel.best_estimator_.get_params()

In [ ]:
best_modellgb = lgb.LGBMRegressor(boosting_type='gbdt',objective='regression',colsample_bytree=1,metrics='auc',learning_rate=lgbmodel.best_estimator_.get_params()['learning_rate'], n_estimators=lgbmodel.best_estimator_.get_params()['n_estimators'], max_depth=lgbmodel.best_estimator_.get_params()['max_depth'],num_leaves = lgbmodel.best_estimator_.get_params()['num_leaves'],max_bin = lgbmodel.best_estimator_.get_params()['max_bin'],min_child_samples=lgbmodel.best_estimator_.get_params()['min_child_samples'],min_child_weight=lgbmodel.best_estimator_.get_params()['min_child_weight'],min_split_gain=lgbmodel.best_estimator_.get_params()['min_split_gain'],min_data_in_leaf = lgbmodel.best_estimator_.get_params()['min_data_in_leaf'], bagging_fraction = lgbmodel.best_estimator_.get_params()['bagging_fraction'],feature_fraction = lgbmodel.best_estimator_.get_params()['feature_fraction'],bagging_freq=lgbmodel.best_estimator_.get_params()['bagging_freq'],lambda_l1=lgbmodel.best_estimator_.get_params()['lambda_l1'],lambda_l2=lgbmodel.best_estimator_.get_params()['lambda_l2'])
  
best_modellgb.fit(X_train_final,y_train)

In [ ]:
import graphviz #安装过程比较困难，耐心
lgb.create_tree_digraph(best_modellgb,tree_index=0,orientation='vertical')

# Pre

In [ ]:
X_test_num = num_pipeline.transform(df_test)
X_test_cate = cat_pipeline.transform(df_test)
X_test_date = date_pipeline.transform(df_test)
df_t = pd.concat([X_test_num, X_test_cate, X_test_date], axis=1)
df_t = df_t[to_keep_final]

In [ ]:
pred=best_modellgb.predict(df_t)
df_pred=pd.DataFrame({'Id':df_test['Id'],'Sold Price': np.exp(pred)})
print(df_pred.head())
df_pred.to_csv('submission.csv', index=False)